## NB02-Data transformation

### Load raw data

In [2]:
import json
import pandas as pd

In [3]:
with open("../data/raw/movies.json", "r") as f:
    all_pages = json.load(f)

with open("../data/raw/genres.json", "r") as f:
    genres_raw = json.load(f)

### Decision: flatten pages

In [4]:
all_movies = []
for page_data in all_pages:
    all_movies.extend(page_data["results"]) #`.extend()` adds all items from a list individually (unlike `.append()`, which would add the whole `results` list as one nested item). 

print(f"Total movie records collected: {len(all_movies)}")

Total movie records collected: 10000




`all_pages` is a list of 50 API responses — one dict per page, each holding
its own `results` list of 20 movies  To analyse movies as a single
table later, we need one flat list of movie records instead of 50 separate
page-dicts.



### Build the movies and genre DataFrames

In [5]:
genres_df = pd.DataFrame(genres_raw["genres"])
genres_df.head()

,id,name
0,28,Action
1,12,Adventure
2,16,Animation
3,35,Comedy
4,80,Crime


In [6]:
movies_df = pd.DataFrame(all_movies)
movies_df.head()

,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,/tYuC9kUwqhpDQ3pv1kLMqyMF1Jw.jpg,"[12, 28, 14]",1368337,The Odyssey,en,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",1116.8359,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg,2026-07-15,False,False,7.965,1702
1,False,/54KIfdTEzOliHDKx0OkzYGqAICx.jpg,"[28, 12, 878]",1081003,Supergirl,en,Supergirl,When an unexpected and ruthless adversary stri...,610.3804,/1QCWdqzTfh2x9UylVpspIU6QTuM.jpg,2026-06-24,False,False,6.670,929
2,False,/piV2OnzTZCyGBP9JCjlHIgKGlfo.jpg,"[28, 14, 878]",454639,Masters of the Universe,en,Masters of the Universe,"After being separated for 15 years, the Sword ...",532.6297,/oRuyGUHdoaQxWP3SDfafGkStxTC.jpg,2026-06-03,False,False,7.298,1260
3,False,/flxau5Iu7bChQHsESqvGZ3FQRaI.jpg,"[878, 53]",1275779,Disclosure Day,en,Disclosure Day,A cybersecurity expert becomes a whistleblower...,498.8136,/AnJ8IQJI23hNpYXVNaythu061Ru.jpg,2026-06-10,False,False,7.405,1956
4,False,/c6BPbkO5Npt1OdwttAxCFo06wtH.jpg,"[10751, 14, 35, 12]",1108427,Moana,en,Moana,"Teenage Moana answers the Ocean's call and, fo...",452.3372,/zKVgiv5qHCvCLT4A2ymJi5QeXDH.jpg,2026-07-08,False,False,5.901,151


### Deleting duplicated values

In [7]:
print("Duplicate movie ids:", movies_df["id"].duplicated().sum())
movies_df = movies_df.drop_duplicates(subset="id")

Duplicate movie ids: 317


### Filtering the data set

Analizing the `vote_count` and `popularity`. TMDB's `popularity` is a platform-calculated score reflecting current interest and activity (page views, recent votes, watchlist/favourite additions, release
recency) — not perceived quality.

In [8]:
print("Minimum vote_count:", movies_df["vote_count"].min(), "/  Mean vote_count:", movies_df["vote_count"].mean(), "/  Maximum vote_count:", movies_df["vote_count"].max())
print("Minimum popularity:", movies_df["popularity"].min(), "/  Mean popularity:", movies_df["popularity"].mean(), "/  Maximum popularity:", movies_df["popularity"].max())

Minimum vote_count: 0 /  Mean vote_count: 1794.2203862439326 /  Maximum vote_count: 40480
Minimum popularity: 1.9243 /  Mean popularity: 8.105571930186926 /  Maximum popularity: 1116.8359


In [9]:
threshold = movies_df["vote_count"].quantile(0.25) #`.quantile(p)` returns the value below which `p`% of the data falls.
print("25th percentile (vote_count):", threshold)
movies_df.sort_values("vote_count").head(10)

25th percentile (vote_count): 1.0


,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
9922,False,None,[35],1522992,Hajtűkanyar Autósiskola,hu,Hajtűkanyar Autósiskola,,3.0777,None,,False,False,0.0,0
9997,False,None,[35],979592,Кавказский тверк,ru,Кавказский тверк,Feature film by Amet Magomedov.,3.7789,None,2022-06-30,False,False,0.0,0
9992,False,None,[],1094829,In Bloom,en,In Bloom,An isolated man dealing with loss and anger is...,3.8116,/qdDoEu9a3nuIs9DAHIi0kURo0bA.jpg,2023-01-30,False,False,0.0,0
9981,False,None,[27],1285108,Eye Socket,en,Eye Socket,"Mary, an agoraphobic woman must survive as her...",3.7936,/erasATRVpcrloxjZi4LDctL8fyO.jpg,2024-05-02,False,False,0.0,0
9989,False,None,[80],1105030,Uso-Tsuki,en,Uso-Tsuki,A man's worst nightmare may just be the truth.,4.3245,/6CKTxOvtfCjAS5mCIQ0CU1CwCCA.jpg,2023-03-25,False,False,0.0,0
9979,False,None,[18],306773,Hati Iblis,ms,Hati Iblis,Aini (Aini Hayati) is left to fend for herself...,3.5577,/puJNG2tRE6IHHNXk68dTQcYKzRh.jpg,1953-06-16,False,False,0.0,0
9975,False,/5b5T050ZNgHB8PVrjFTvWTnb3Vx.jpg,"[53, 9648]",1096873,Bitter Fruit,pt,Fruto Amargo,"Jorge, a young widower, visits his wife's grav...",4.3270,/cTCksosFQsF5vF4hgwohi9Ol5vJ.jpg,,False,False,0.0,0
78,False,/v2SOWxOgifIhnZFLgIEvzr7ZbJ2.jpg,[27],1567187,A Mother's Recall,es,Memoria de una madre,A married couple welcomes young Genaro into th...,55.6586,/vhfrAIIe9jHiuD9qjOhLwnEgLdb.jpg,2025-10-21,False,False,0.0,0
6783,False,None,[18],1566141,Surface Tension,tl,Surface Tension,Competitive swimmer Bulet finds herself drowni...,5.1472,/msLKKsDidpRiO9mdu6sPSZpyvuu.jpg,2025-11-14,False,False,0.0,0
6786,False,/2wYjHAvgUVWYh9Y46xXQ5RgM5BH.jpg,"[18, 35, 99]",1455712,Identidade,pt,Identidade,During a screening of Oedipus Rex in a run-dow...,5.0939,/j0zqTt7EABV3pYQ7Mw8w918v7Ij.jpg,2026-04-01,False,False,0.0,0



Decision: The 25th percentile of `vote_count` was 312.5 votes. I dropped rows below this threshold instead
of picking an arbitrary cutoff to maximize the reliability of the data.

In [10]:
print("Before filtering:", len(movies_df))
movies_df = movies_df[movies_df["vote_count"] > threshold]
print("After filtering:", len(movies_df))

Before filtering: 9683
After filtering: 7257


Analizyng bool columns 

In [11]:
bool_columns = movies_df.select_dtypes(include="bool").columns # `.select_dtypes(include="x")` returns only the columns whose data type is x.
for col in bool_columns:
    print(f"--- {col} ---")
    print(movies_df[col].value_counts())
    print()

--- adult ---
adult
False    7257
Name: count, dtype: int64

--- softcore ---
softcore
False    7257
Name: count, dtype: int64

--- video ---
video
False    7257
Name: count, dtype: int64



Decision: drop boolean columns. `Adult` , `softcore` and `video` showed  no variation ( all False), so they add no useful information.

In [12]:
movies_df = movies_df.drop(columns=bool_columns)
movies_df.columns

Index(['backdrop_path', 'genre_ids', 'id', 'title', 'original_language',
       'original_title', 'overview', 'popularity', 'poster_path',
       'release_date', 'vote_average', 'vote_count'],
      dtype='object')

Decision: drop columns not needed for the research question. `backdrop_path`, `poster_path` are image URLs (not analysable data). `overview` is free text, not needed for a genre/rating/popularity comparison. `original_title` is redundant with `title` for this analysis.


In [13]:
movies_df = movies_df.drop(columns=["backdrop_path", "poster_path", "overview", "original_title"])
movies_df.columns

Index(['genre_ids', 'id', 'title', 'original_language', 'popularity',
       'release_date', 'vote_average', 'vote_count'],
      dtype='object')

Decision: extract release year to compare "over the years". 

In [14]:
movies_df["release_date"] = pd.to_datetime(movies_df["release_date"])
movies_df["release_year"] = movies_df["release_date"].dt.year
movies_df.head()

,genre_ids,id,title,original_language,popularity,release_date,vote_average,vote_count,release_year
0,"[12, 28, 14]",1368337,The Odyssey,en,1116.8359,2026-07-15,7.965,1702,2026.0
1,"[28, 12, 878]",1081003,Supergirl,en,610.3804,2026-06-24,6.670,929,2026.0
2,"[28, 14, 878]",454639,Masters of the Universe,en,532.6297,2026-06-03,7.298,1260,2026.0
3,"[878, 53]",1275779,Disclosure Day,en,498.8136,2026-06-10,7.405,1956,2026.0
4,"[10751, 14, 35, 12]",1108427,Moana,en,452.3372,2026-07-08,5.901,151,2026.0


Decision: add a column od decade, to simplify historical evolution

In [23]:
movies_df["decade"] = (movies_df["release_year"] // 10 * 10).astype(int)

### Search for null rows

In [24]:
for name, df in [("movies_df", movies_df), ("genres_df", genres_df)]:
    print(f"--- {name} ---")
    print(df.isna().sum())  # counts nulls per column
    print()

--- movies_df ---
genre_ids            0
id                   0
title                0
original_language    0
popularity           0
release_date         0
vote_average         0
vote_count           0
release_year         0
decade               0
dtype: int64

--- genres_df ---
id      0
name    0
dtype: int64



Decision: drop rows with null values

In [18]:
movies_df = movies_df.dropna()
genres_df = genres_df.dropna()

### Decision: merge the tables genre and movies

`genre_ids` are just numbers, merging with the genre lookup table replaces
them with real genre names, which makes posible grouping them by genre names for future exploration. Aditionally, I change the name of the column `name` for `genre`

In [25]:
movies_exploded = movies_df.explode("genre_ids") #genre_ids is a list per movie, .explode() turns each list item into its own row
movies_with_genres = movies_exploded.merge(
    genres_df,
    left_on="genre_ids",
    right_on="id",
    how="left",
    suffixes=("", "_genre")
)
movies_with_genres = movies_with_genres.drop(columns=["id_genre"])
movies_with_genres.columns
movies_with_genres = movies_with_genres.rename(columns={"name": "genre"})

movies_with_genres.head()

,genre_ids,id,title,original_language,popularity,release_date,vote_average,vote_count,release_year,decade,genre
0,12,1368337,The Odyssey,en,1116.8359,2026-07-15,7.965,1702,2026.0,2020,Adventure
1,28,1368337,The Odyssey,en,1116.8359,2026-07-15,7.965,1702,2026.0,2020,Action
2,14,1368337,The Odyssey,en,1116.8359,2026-07-15,7.965,1702,2026.0,2020,Fantasy
3,28,1081003,Supergirl,en,610.3804,2026-06-24,6.670,929,2026.0,2020,Action
4,12,1081003,Supergirl,en,610.3804,2026-06-24,6.670,929,2026.0,2020,Adventure


### Save the prepared data 

In [26]:
movies_df.to_csv("../data/processed/movies.csv", index=False)
genres_df.to_csv("../data/processed/genres.csv", index=False)
movies_with_genres.to_csv("../data/processed/movies_with_genres.csv", index=False)